In [3]:
import pandas as pd
import numpy as np

In [4]:
def aplicar_conformal_prediction(csv_path, alpha=0.05):
    # 1. Carregar os resultados da auditoria
    df = pd.read_csv(csv_path)
    
    # 2. Particionamento (Ex: 50% Calibração, 50% Teste)
    # n_calib deve ser pelo menos 100 para validade mínima
    df_calib = df.sample(frac=0.5, random_state=42)
    df_test = df.drop(df_calib.index)
    
    n = len(df_calib)
    scores_calib = df_calib['non_conformity_score'].values
    
    # 3. Cálculo do Quantil Conformal (Threshold)
    # Fórmula: (n+1)(1-alpha)/n
    q_level = np.ceil((n + 1) * (1 - alpha)) / n
    q_hat = np.quantile(scores_calib, q_level, method='higher')
    
    print(f"--- RESULTADOS ESTATÍSTICOS ---")
    print(f"Tamanho da amostra de calibração (n): {n}")
    print(f"Nível de significância (alpha): {alpha}")
    print(f"Limiar calculado (q_hat): {q_hat}")
    
    # 4. Aplicação no Conjunto de Teste (Inferência com Garantia)
    df_test['aprovado_pelo_guardrail'] = df_test['non_conformity_score'] <= q_hat
    
    # 5. Métricas para o Artigo
    precisao_final = df_test[df_test['aprovado_pelo_guardrail'] == True]['non_conformity_score'].mean()
    taxa_retencao = df_test['aprovado_pelo_guardrail'].mean()
    
    print(f"Taxa de Fatos Filtrados (Retenção): {taxa_retencao*100:.2f}%")
    print(f"Incerteza Residual Estimada: {precisao_final*100:.2f}%")
    
    df_test.to_csv("resultados_finais_conformal.csv", index=False)
    return q_hat



In [5]:
q_limiar = aplicar_conformal_prediction("dados_brutos_experimento.csv")

--- RESULTADOS ESTATÍSTICOS ---
Tamanho da amostra de calibração (n): 132
Nível de significância (alpha): 0.05
Limiar calculado (q_hat): 1.0
Taxa de Fatos Filtrados (Retenção): 100.00%
Incerteza Residual Estimada: 30.83%
